# Tokenizer Data-Mixing Study: Which Domain Mix Gives the Best BPE Tokenizer?

The Phase 1 data has **4 training files** (domains). The tokenizer used so far was trained on the
Supreme Court file only. This notebook trains one byte-level BPE tokenizer per **mixing strategy**
(how much text to take from each domain) and compares them by **fertility**.

**Fertility = tokens per word** (words = whitespace-separated, the same as the 128-word evaluation).
Lower is better: each word costs fewer tokens, so the model sees more text per 1024-token block, fewer
prompts get truncated, and 128 words need fewer generation steps. All tokenizers get the **same vocabulary
size and the same number of training characters**, so the only difference is the mix.

**Strategies compared**
| Strategy | Share of the training characters from domain *d* |
|---|---|
| `only_<domain>` (x4) | 100% from one domain (`only_supreme_court` = what we use now) |
| `uniform` | equal share from every domain (alpha = 0) |
| `temperature_a0.3`, `temperature_a0.5` | proportional to size^alpha (between uniform and proportional) |
| `proportional` | proportional to the domain's size (alpha = 1), like concatenating all files |
| `test_matched` | proportional to how many **test prompts** look like each domain (unigram classifier) |
| `anchored_<p>` (40/50/60) | p% from the Supreme Court file, the rest split equally over the other 3 domains |

**Measured** on held-out documents from each domain (never used for tokenizer training) and on the
test prompts: fertility, characters per token, % of words split into 2+ tokens, mean prompt length in
tokens, and % of test prompts longer than the 640-token prompt window (they get truncated).

**Choosing the final tokenizer.** The test set may be more varied than our estimate, so the winner is
not simply the lowest test fertility. Among the strategies that cost at most `MAX_TEST_COST_PCT` (1%)
more tokens on the test prompts than the best one, the notebook picks the one with the **lowest
average fertility over the 4 domains** (the most robust). That mix is then **retrained on
`FINAL_TRAIN_CHARS` (300M) characters**, the same budget as our current tokenizer, and saved as
`best_tokenizer/bpe_tokenizer.json`, ready for the training notebooks.

**Run on Kaggle:** Accelerator **None (CPU)** is enough (BPE training is pure Python CPU work);
**Internet on**; attach the **`HF`** secret; then **Save Version -> Save & Run All (Commit)**.

**Smoke test first:** `SMOKE_TEST = True` (default) runs every step at tiny scale (2k vocab, 2M characters
per tokenizer) in roughly 10-20 minutes (mostly downloading the 4 files), writing to
`/kaggle/working/cs781_tokenizer_mixing_smoke/`. If it ends with `SMOKE TEST PASSED`, set
`SMOKE_TEST = False` and commit again; results go to `/kaggle/working/cs781_tokenizer_mixing/`.

## 1. Hugging Face Login

All imports are here. The token is read from Kaggle Secrets (name `HF`, which must be **attached** to
this notebook).

In [ ]:
import os, json, math, time, random, heapq, gc, shutil, contextlib
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import login, hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")
print("CPU cores:", os.cpu_count())

## 2. Configuration

- `VOCAB_SIZE` / `TOKENIZER_TRAIN_CHARS`: identical for every tokenizer, so the comparison is fair.
  50k matches the vocabulary of our models. 50M characters per tokenizer keeps the whole study to a few
  hours; fertility changes little beyond that, and the winning mix can be retrained on more text later.
- `ALPHAS`: temperature-mixing exponents (0 = uniform, 1 = proportional).
- `ANCHOR_DOMAIN` / `ANCHOR_SHARES`: the `anchored_<p>` strategies (p% from the anchor domain).
- `MAX_TEST_COST_PCT`: how many % extra test-prompt tokens we accept in exchange for robustness.
- `FINAL_STRATEGY`: `None` = chosen automatically by the rule above; or force one by name.
- `FINAL_TRAIN_CHARS`: training characters of the final, retrained tokenizer.
- `EVAL_CHARS_PER_DOMAIN`: size of each domain's held-out evaluation set.
- `N_WORKERS`: tokenizers trained in parallel (one per CPU core; each needs a few GB of RAM).

In [ ]:
# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_DIR = "Phase_1_Pretraining/Train/"
TRAIN_FILES = None                    # None = every .json file in TRAIN_DIR (listed from the repo)
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Tokenizers (the same for every strategy -> fair comparison)
VOCAB_SIZE = 50000
TOKENIZER_TRAIN_CHARS = 50_000_000

# Strategies
INCLUDE_SINGLE_DOMAIN = True          # only_<domain> for each of the 4 domains
ALPHAS = [0.0, 0.3, 0.5, 1.0]         # share_d ~ size_d ** alpha  (0 = uniform, 1 = proportional)
INCLUDE_TEST_MATCHED = True           # share_d ~ fraction of test prompts that look like domain d
ANCHOR_DOMAIN = "supreme_court"       # ~80% of the test prompts look like this domain
ANCHOR_SHARES = [0.4, 0.5, 0.6]       # anchored_<p>: p from ANCHOR_DOMAIN, the rest split equally

# Final tokenizer
MAX_TEST_COST_PCT = 1.0               # accept up to 1% more test-prompt tokens than the best strategy ...
                                      # ... and among those pick the lowest average fertility over the 4 domains
FINAL_STRATEGY = None                 # None = automatic; or a name, e.g. "anchored_50" or "uniform"
FINAL_TRAIN_CHARS = 300_000_000       # the chosen mix is retrained on this many characters

# Evaluation
EVAL_CHARS_PER_DOMAIN = 2_000_000     # held-out documents per domain (never used for training)
NUM_TEST_PROMPTS = None               # None = all test prompts
PROMPT_WINDOW_TOKENS = 640            # prompt budget of our models (1024 context - 384 generation)

N_WORKERS = min(4, os.cpu_count() or 1)
SEED = 42
OUTPUT_DIR = "/kaggle/working/cs781_tokenizer_mixing"

# ---- SMOKE TEST: True = tiny run (~10-20 min) to check everything works. False = the real study.
SMOKE_TEST = True
if SMOKE_TEST:
    VOCAB_SIZE, TOKENIZER_TRAIN_CHARS, FINAL_TRAIN_CHARS = 2000, 2_000_000, 4_000_000
    EVAL_CHARS_PER_DOMAIN, NUM_TEST_PROMPTS = 200_000, 200
    OUTPUT_DIR = "/kaggle/working/cs781_tokenizer_mixing_smoke"   # separate folder from the real run
    print("*** SMOKE TEST MODE: tiny run, results are NOT meaningful ***")

TOKENIZERS_DIR = os.path.join(OUTPUT_DIR, "tokenizers")
BEST_TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "best_tokenizer")
RESULTS_CSV = os.path.join(OUTPUT_DIR, "fertility_results.csv")
RESULTS_JSON = os.path.join(OUTPUT_DIR, "fertility_results.json")
HEATMAP_PATH = os.path.join(OUTPUT_DIR, "fertility_heatmap.png")
BARS_PATH = os.path.join(OUTPUT_DIR, "fertility_test_prompts.png")
os.makedirs(TOKENIZERS_DIR, exist_ok=True)
print("Configuration loaded. Outputs go to:", OUTPUT_DIR)

## 3. Byte-Level BPE Tokenizer

The **same `ByteLevelBPE` class** as in our training notebooks, unchanged, so every tokenizer saved here
(`bpe_tokenizer.json`) can be dropped straight into them (`LegalBPETokenizer` loads it).

In [ ]:
_SHIFT = 20
_MASK = (1 << _SHIFT) - 1
_EOW = _MASK


class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.merges = []
        self.symbol_to_id = {}
        self._ranks = None
        self._cache = {}

    def fit(self, sentences):
        word_freq = Counter()
        for s in sentences:
            word_freq.update(s.split())

        sym = {}
        key = {}
        for b in range(256):
            i = len(self.itos)
            self.symbol_to_id[(b,)] = i
            self.itos.append([b])
            sym[i] = (b,)
            key[i] = chr(b)
        sym[_EOW] = (256,)
        key[_EOW] = chr(256)

        words, freqs = [], []
        for w, f in word_freq.items():
            words.append([b + 4 for b in w.encode("utf-8")] + [_EOW])
            freqs.append(f)
        del word_freq

        pc = defaultdict(int)
        where = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for x, y in zip(w, w[1:]):
                p = (x << _SHIFT) | y
                pc[p] += f
                where[p].add(wi)

        heap = [(-c, key[p >> _SHIFT], key[p & _MASK], p) for p, c in pc.items() if c > 1]
        heapq.heapify(heap)

        itos, merges, s2id = self.itos, self.merges, self.symbol_to_id
        target = self.target_vocab_size
        heappop, heappush = heapq.heappop, heapq.heappush

        while len(itos) < target:
            while heap:
                negc, _, _, best = heappop(heap)
                if pc.get(best, 0) == -negc:
                    break
            else:
                break
            if -negc <= 1:
                break

            a, b = best >> _SHIFT, best & _MASK
            new = len(itos)
            new_sym = sym[a] + sym[b]
            merges.append([list(sym[a]), list(sym[b])])
            s2id[new_sym] = new
            itos.append(list(new_sym))
            sym[new] = new_sym
            key[new] = key[a] + key[b]

            changed = set()
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                i = 0
                while i < len(w) - 1:
                    if w[i] == a and w[i + 1] == b:
                        if i > 0:
                            l = w[i - 1]
                            p = (l << _SHIFT) | a; pc[p] -= f; changed.add(p)
                            p = (l << _SHIFT) | new; pc[p] += f; changed.add(p); where[p].add(wi)
                        if i + 2 < len(w):
                            r = w[i + 2]
                            p = (b << _SHIFT) | r; pc[p] -= f; changed.add(p)
                            p = (new << _SHIFT) | r; pc[p] += f; changed.add(p); where[p].add(wi)
                        pc[best] -= f
                        w[i:i + 2] = [new]
                    i += 1

            pc.pop(best, None)
            changed.discard(best)
            for p in changed:
                c = pc[p]
                if c > 1:
                    heappush(heap, (-c, key[p >> _SHIFT], key[p & _MASK], p))
                elif c <= 0:
                    del pc[p]

            if len(merges) % 500 == 0:
                print(f"Merges: {len(merges):,} | Vocab: {len(itos):,} | Best pair count: {-negc:,}")

        self.vocab_size = len(itos)
        self._ranks = None
        self._cache = {}
        print(f"BPE trained: {self.vocab_size:,} vocabulary entries, {len(merges):,} merges")
        return self

    def _build_ranks(self):
        def to_id(t):
            t = tuple(t)
            return _EOW if t == (256,) else self.symbol_to_id[t]
        if not self.symbol_to_id:
            for i, s in enumerate(self.itos):
                if isinstance(s, list):
                    self.symbol_to_id[tuple(s)] = i
        self._ranks = {}
        for l, r in self.merges:
            self._ranks[(to_id(l) << _SHIFT) | to_id(r)] = self.symbol_to_id[tuple(l) + tuple(r)]

    def tokenize_word(self, word):
        out = self._cache.get(word)
        if out is not None:
            return out
        if self._ranks is None:
            self._build_ranks()
        ranks = self._ranks
        w = [b + 4 for b in word.encode("utf-8")]
        w.append(_EOW)

        while len(w) > 1:
            best_new, best_p = None, None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best_new is None or n < best_new):
                    best_new, best_p = n, (x, y)
            if best_new is None:
                break
            a, b = best_p
            i, res = 0, []
            L = len(w)
            while i < L:
                if i < L - 1 and w[i] == a and w[i + 1] == b:
                    res.append(best_new); i += 2
                else:
                    res.append(w[i]); i += 1
            w = res

        out = [1 if t == _EOW else t for t in w]
        self._cache[word] = out
        return out

    def encode_sentence(self, sentence):
        ids = []
        tw = self.tokenize_word
        for word in sentence.split():
            ids.extend(tw(word))
        return ids

    def encode_corpus(self, sentences):
        return [self.encode_sentence(s) for s in sentences]

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos,
                       "vocab_size": self.vocab_size}, f, ensure_ascii=False)

## 4. Download the 4 Training Files and Split Each Domain

The training files are several GB each, and `json.load` of a whole file can need several times its
size in RAM, which kills the Kaggle kernel. So each file is **streamed** (`iter_json_records` reads one
record at a time with the standard `json` decoder) in two passes:
1. **Count pass:** number of documents and characters in the domain (the size-based strategies need it).
2. **Sample pass:** keeps each document with a fixed-seed probability, chosen so that about 1.5x the
   needed text is kept (`EVAL_CHARS_PER_DOMAIN + POOL_CHARS`). The kept documents are shuffled;
   the first `EVAL_CHARS_PER_DOMAIN` characters become the domain's **held-out evaluation set** and the
   next `POOL_CHARS` characters its **training pool** (every strategy samples from the pools).
   `POOL_CHARS = max(TOKENIZER_TRAIN_CHARS, FINAL_TRAIN_CHARS)`, so the final retrain has enough text;
   the comparison tokenizers use only the start of each pool, exactly as before.

Only the sample is held in memory. RAM usage is printed after every file.

In [ ]:
def iter_json_records(path, chunk_size=1 << 24):
    """Yield the items of a top-level JSON list (or the values of a top-level JSON object)
    one at a time, without loading the whole file into memory."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf, pos, eof = "", 0, False

        def fill():
            nonlocal buf, pos, eof
            chunk = f.read(chunk_size)
            eof = not chunk
            buf, pos = buf[pos:] + chunk, 0

        def skip(chars):                              # skip whitespace and the given separators
            nonlocal pos
            while True:
                while pos < len(buf) and (buf[pos].isspace() or buf[pos] in chars):
                    pos += 1
                if pos < len(buf) or eof:
                    return
                fill()

        def decode():
            nonlocal pos
            while True:
                try:
                    obj, pos = dec.raw_decode(buf, pos)
                    return obj
                except json.JSONDecodeError:
                    if eof:
                        raise
                    fill()                            # the item continues in the next chunk

        skip("")
        is_dict = buf[pos] == "{"
        pos += 1
        while True:
            skip(",")
            if pos >= len(buf) or buf[pos] in "]}":
                return
            if is_dict:
                decode()                              # the key (an id)
                skip(":")
            yield decode()


def record_text(r):
    t = r.get("text") if isinstance(r, dict) else None
    return t if isinstance(t, str) and t.strip() else None


def ram_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**30
    except Exception:
        return float("nan")


if TRAIN_FILES is None:
    TRAIN_FILES = sorted(f for f in HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
                         if f.startswith(TRAIN_DIR) and f.endswith(".json"))
print("Training files:", TRAIN_FILES)
assert TRAIN_FILES, "no training files found"


def domain_name(path):
    name = os.path.basename(path)[:-len(".json")]
    return name[len("train_"):] if name.startswith("train_") else name


POOL_CHARS = max(TOKENIZER_TRAIN_CHARS, FINAL_TRAIN_CHARS)
train_pool, eval_texts, domain_stats = {}, {}, {}
for path in TRAIN_FILES:
    t0 = time.time()
    d = domain_name(path)
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

    # pass 1: size of the domain
    n_docs = n_chars = 0
    for r in iter_json_records(local_path):
        t = record_text(r)
        if t:
            n_docs += 1
            n_chars += len(t)

    # pass 2: keep a random ~1.5x sample of the text we need
    need = EVAL_CHARS_PER_DOMAIN + POOL_CHARS
    keep_p = min(1.0, 1.5 * need / max(n_chars, 1))
    rng = random.Random(f"{SEED}-{d}")
    kept = []
    for r in iter_json_records(local_path):
        t = record_text(r)
        if t and rng.random() < keep_p:
            kept.append(t)
    rng.shuffle(kept)

    ev, tr, ev_chars, tr_chars = [], [], 0, 0
    for t in kept:
        if ev_chars < EVAL_CHARS_PER_DOMAIN:
            ev.append(t); ev_chars += len(t)
        elif tr_chars < POOL_CHARS:
            tr.append(t); tr_chars += len(t)
        else:
            break
    train_pool[d], eval_texts[d] = tr, ev
    domain_stats[d] = {"file": path, "file_GB": os.path.getsize(local_path) / 2**30, "documents": n_docs,
                       "characters": n_chars, "eval_characters": ev_chars, "pool_characters": tr_chars}
    del kept
    gc.collect()
    print(f"{d:25s} {os.path.getsize(local_path) / 2**30:5.2f} GB file | {n_docs:>9,} docs  {n_chars / 1e6:>9,.1f}M chars  "
          f"| eval {ev_chars / 1e6:.1f}M | pool {tr_chars / 1e6:.1f}M | RAM {ram_gb():.1f} GB  ({time.time() - t0:.0f}s)")
DOMAINS = list(train_pool)
for d in DOMAINS:
    if domain_stats[d]["pool_characters"] < POOL_CHARS:
        print(f"NOTE: {d} has only {domain_stats[d]['pool_characters'] / 1e6:.1f}M characters available for "
              f"training (< {POOL_CHARS / 1e6:.0f}M); strategies needing more use all of it.")
print("\nDomain sizes:")
print(pd.DataFrame(domain_stats).T.drop(columns=["file"]).to_string())

## 5. Load the Test Prompts

Fertility on the **test prompts** is the most relevant number: these are the texts our models have to
read and continue.

In [ ]:
test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_texts = [r["text"] for r in test_data.values() if isinstance(r, dict) and isinstance(r.get("text"), str)]
if NUM_TEST_PROMPTS:
    test_texts = test_texts[:NUM_TEST_PROMPTS]
print(f"Test prompts: {len(test_texts):,} | mean {sum(len(t.split()) for t in test_texts) / len(test_texts):.0f} words")

## 6. Build the Evaluation Sets

Every evaluation set is stored as a **word-frequency table** (`Counter`): a tokenizer splits each word
on its own, so tokens(set) = sum over words of frequency x tokens(word). This makes evaluating a
tokenizer fast (each distinct word is tokenized once).

Evaluation sets: `heldout_<domain>` for each domain, plus `test_prompts`.

In [ ]:
eval_sets = {f"heldout_{d}": Counter(w for t in eval_texts[d] for w in t.split()) for d in DOMAINS}
eval_sets["test_prompts"] = Counter(w for t in test_texts for w in t.split())
test_prompt_words = [t.split() for t in test_texts]          # per-prompt token counts (truncation check)
for name, c in eval_sets.items():
    print(f"{name:30s} {sum(c.values()):>11,} words  {len(c):>9,} distinct")

## 7. Estimate the Test-Set Domain Mix (for `test_matched`)

A simple **unigram (bag-of-words) classifier**: a word-frequency model per domain (built from ~5M
characters of its training pool, add-one smoothing) scores each test prompt, and the prompt is assigned
to the most likely domain. The fraction of test prompts per domain becomes the `test_matched` mix. It
also tells us **which domains the test set actually comes from**, which is useful for choosing the
training data of the language model too.

In [ ]:
def unigram_counts(texts, max_chars=5_000_000):
    c, n = Counter(), 0
    for t in texts:
        c.update(t.lower().split())
        n += len(t)
        if n >= max_chars:
            break
    return c


uni = {d: unigram_counts(train_pool[d]) for d in DOMAINS}
V = len(set().union(*uni.values()))
uni_total = {d: sum(c.values()) for d, c in uni.items()}


def most_likely_domain(text):
    words = text.lower().split()
    scores = {d: sum(math.log((uni[d][w] + 1) / (uni_total[d] + V)) for w in words) for d in DOMAINS}
    return max(scores, key=scores.get)


test_assign = Counter(most_likely_domain(t) for t in test_texts)
test_weights = {d: test_assign[d] / len(test_texts) for d in DOMAINS}
del uni
print("Estimated domain of the test prompts:")
for d in DOMAINS:
    print(f"  {d:25s} {test_assign[d]:>6,} prompts ({100 * test_weights[d]:.1f}%)")

## 8. Define the Mixing Strategies

Each strategy is a set of **shares** (summing to 1): tokenizer training text = share_d x
`TOKENIZER_TRAIN_CHARS` characters from domain *d*. Temperature mixing `share_d ~ size_d ** alpha`
interpolates between uniform (alpha = 0: small domains up-weighted) and proportional (alpha = 1: big
domains dominate). `anchored_<p>` keeps p% from `ANCHOR_DOMAIN` (where most test prompts come from) and
spreads the rest equally, so every domain's vocabulary is represented.

In [ ]:
sizes = {d: domain_stats[d]["characters"] for d in DOMAINS}
strategies = {}
if INCLUDE_SINGLE_DOMAIN:
    for d in DOMAINS:
        strategies[f"only_{d}"] = {x: float(x == d) for x in DOMAINS}
for a in ALPHAS:
    raw = {d: sizes[d] ** a for d in DOMAINS}
    z = sum(raw.values())
    name = "uniform" if a == 0 else "proportional" if a == 1 else f"temperature_a{a}"
    strategies[name] = {d: raw[d] / z for d in DOMAINS}
if INCLUDE_TEST_MATCHED:
    strategies["test_matched"] = dict(test_weights)
if ANCHOR_DOMAIN in DOMAINS:
    for p in ANCHOR_SHARES:
        rest = (1 - p) / (len(DOMAINS) - 1)
        strategies[f"anchored_{round(100 * p)}"] = {d: p if d == ANCHOR_DOMAIN else rest for d in DOMAINS}
elif ANCHOR_SHARES:
    print(f"NOTE: ANCHOR_DOMAIN {ANCHOR_DOMAIN!r} is not one of {DOMAINS}; anchored strategies skipped.")

print(f"{len(strategies)} strategies (share of training characters, %):")
print((pd.DataFrame(strategies).T * 100).round(1).to_string())

## 9. Train and Evaluate One Tokenizer per Strategy (in parallel)

`train_and_evaluate(name)`:
1. Streams the strategy's text from the domain pools (share_d x budget characters per domain).
2. Trains `ByteLevelBPE(VOCAB_SIZE)` and saves `tokenizers/<strategy>/bpe_tokenizer.json` (the merge
   progress log goes to `train_log.txt` next to it).
3. Measures every evaluation set:
   - **fertility** = tokens / words (lower = better)
   - **chars_per_token** = characters / tokens (higher = better compression)
   - **pct_words_split** = % of words that need 2+ tokens
   - on the test prompts also: mean prompt length in tokens and % of prompts longer than
     `PROMPT_WINDOW_TOKENS` (these get truncated by our models)

Strategies run in parallel worker processes (`fork`, so the pools and evaluation sets are shared, not
copied up front). Each finished tokenizer prints one line.

In [ ]:
def evaluate_counter(bpe, counter):
    words = tokens = chars = split = 0
    for w, f in counter.items():
        n = len(bpe.tokenize_word(w))
        words += f
        tokens += f * n
        chars += f * len(w)
        split += f * (n > 1)
    return {"fertility": tokens / words, "chars_per_token": chars / tokens, "pct_words_split": 100 * split / words}


def train_and_evaluate(name, train_chars=None, out_dir=None):
    """Train the strategy's tokenizer on train_chars characters (default TOKENIZER_TRAIN_CHARS),
    save it to out_dir (default tokenizers/<name>/) and measure it on every evaluation set."""
    weights = strategies[name]
    train_chars = train_chars or TOKENIZER_TRAIN_CHARS
    used = Counter()

    def texts():
        for d, share in weights.items():
            need, got = share * train_chars, 0
            for t in train_pool[d]:
                if got >= need:
                    break
                yield t
                got += len(t)
                used[d] += len(t)

    out_dir = out_dir or os.path.join(TOKENIZERS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    with open(os.path.join(out_dir, "train_log.txt"), "w") as log, contextlib.redirect_stdout(log):
        bpe = ByteLevelBPE(vocab_size=VOCAB_SIZE).fit(texts())
    train_minutes = (time.time() - t0) / 60
    bpe.save(os.path.join(out_dir, "bpe_tokenizer.json"))

    result = {"strategy": name, "shares": weights, "chars_used": dict(used),
              "train_chars": sum(used.values()), "vocab_size": bpe.vocab_size,
              "train_minutes": train_minutes, "metrics": {}}
    for set_name, counter in eval_sets.items():
        result["metrics"][set_name] = evaluate_counter(bpe, counter)
    lengths = [sum(len(bpe.tokenize_word(w)) for w in words) for words in test_prompt_words]
    result["test_prompt_tokens_mean"] = sum(lengths) / len(lengths)
    result["pct_test_prompts_truncated"] = 100 * sum(n > PROMPT_WINDOW_TOKENS for n in lengths) / len(lengths)
    result["minutes_total"] = (time.time() - t0) / 60
    return result


names = list(strategies)
results = []
start = time.time()
try:
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=min(N_WORKERS, len(names)), mp_context=ctx) as pool:
        futures = {pool.submit(train_and_evaluate, n): n for n in names}
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            print(f"[{len(results)}/{len(names)}] {r['strategy']:28s} test fertility {r['metrics']['test_prompts']['fertility']:.4f} "
                  f"| trained in {r['train_minutes']:.1f} min | elapsed {(time.time() - start) / 60:.1f} min")
except Exception as e:                                    # e.g. a worker ran out of memory
    print(f"Parallel run failed ({type(e).__name__}: {e}); falling back to one tokenizer at a time.")
    done = {r["strategy"] for r in results}
    for n in names:
        if n not in done:
            r = train_and_evaluate(n)
            results.append(r)
            print(f"[{len(results)}/{len(names)}] {n:28s} test fertility {r['metrics']['test_prompts']['fertility']:.4f}")
print(f"All tokenizers done in {(time.time() - start) / 60:.1f} min")

## 10. Results Table

One row per strategy, sorted by **fertility on the test prompts** (the main criterion).
- `fert_<set>`: fertility on each evaluation set.
- `fert_heldout_macro`: average over the 4 held-out domains (robustness across domains).
- `fert_heldout_worst`: the worst domain.
- `test_cost_pct`: % more test-prompt tokens than the best strategy (0 = the best one).
- `test_prompt_tokens`, `pct_truncated`: mean test prompt length in tokens, and the % of prompts that
  do not fit the 640-token prompt window.
- `tokens_for_128_words`: about how many tokens a 128-word continuation needs (vs. `MAX_GENERATION_TOKENS = 384`).

Saved as `fertility_results.csv` (table) and `fertility_results.json` (everything, including the
shares and the characters actually used per domain).

In [ ]:
rows = []
for r in results:
    m = r["metrics"]
    row = {"strategy": r["strategy"]}
    for set_name in eval_sets:
        row[f"fert_{set_name}"] = m[set_name]["fertility"]
    held = [m[f"heldout_{d}"]["fertility"] for d in DOMAINS]
    row["fert_heldout_macro"] = sum(held) / len(held)
    row["fert_heldout_worst"] = max(held)
    row["chars_per_token_test"] = m["test_prompts"]["chars_per_token"]
    row["pct_words_split_test"] = m["test_prompts"]["pct_words_split"]
    row["test_prompt_tokens"] = r["test_prompt_tokens_mean"]
    row["pct_truncated"] = r["pct_test_prompts_truncated"]
    row["tokens_for_128_words"] = 128 * m["test_prompts"]["fertility"]
    row["vocab_size"] = r["vocab_size"]
    row["train_chars_M"] = r["train_chars"] / 1e6
    row["train_minutes"] = r["train_minutes"]
    rows.append(row)

df = pd.DataFrame(rows).sort_values("fert_test_prompts").reset_index(drop=True)
df.insert(1, "test_cost_pct", 100 * (df["fert_test_prompts"] / df["fert_test_prompts"].min() - 1))
df.to_csv(RESULTS_CSV, index=False)
with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump({"config": {"vocab_size": VOCAB_SIZE, "tokenizer_train_chars": TOKENIZER_TRAIN_CHARS,
                          "eval_chars_per_domain": EVAL_CHARS_PER_DOMAIN, "smoke_test": SMOKE_TEST},
               "domain_stats": domain_stats, "test_domain_estimate": test_weights,
               "results": sorted(results, key=lambda r: r["metrics"]["test_prompts"]["fertility"])}, f, indent=2)
print("Saved:", RESULTS_CSV)
print("Saved:", RESULTS_JSON)
pd.set_option("display.width", 250)
print("\nRESULTS (sorted by test-prompt fertility, lower = better):")
print(df.round(4).to_string())
print("\nFertility per evaluation set:")
print(df.set_index("strategy")[[c for c in df.columns if c.startswith("fert_")]].round(4).to_string())

## 11. Plots

- **Heatmap:** fertility of every tokenizer (rows) on every evaluation set (columns). The diagonal-like
  pattern of the `only_<domain>` rows shows how much a single-domain tokenizer loses on other domains.
- **Bar chart:** fertility on the test prompts, best first. The dashed line is our current choice
  (`only_supreme_court`); the dotted line is the `MAX_TEST_COST_PCT` limit (strategies left of it are
  candidates for the final tokenizer).

In [ ]:
set_cols = [f"fert_{s}" for s in eval_sets]
mat = df.set_index("strategy")[set_cols]
fig, ax = plt.subplots(figsize=(1.8 * len(set_cols) + 3, 0.5 * len(mat) + 2))
im = ax.imshow(mat.values, cmap="viridis_r", aspect="auto")
ax.set_xticks(range(len(set_cols)), [c[len("fert_"):] for c in set_cols], rotation=30, ha="right")
ax.set_yticks(range(len(mat)), mat.index)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat.values[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
fig.colorbar(im, ax=ax, label="fertility (tokens / word, lower = better)")
ax.set_title(f"Tokenizer fertility by mixing strategy (vocab {VOCAB_SIZE:,}, {TOKENIZER_TRAIN_CHARS / 1e6:.0f}M chars each)")
plt.tight_layout()
plt.savefig(HEATMAP_PATH, dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(10, 0.45 * len(df) + 1.5))
ax.barh(df["strategy"][::-1], df["fert_test_prompts"][::-1], color="tab:blue")
if "only_supreme_court" in set(df["strategy"]):
    ax.axvline(df.loc[df["strategy"] == "only_supreme_court", "fert_test_prompts"].item(),
               color="tab:red", ls="--", label="only_supreme_court (current)")
ax.axvline(df["fert_test_prompts"].min() * (1 + MAX_TEST_COST_PCT / 100), color="gray", ls=":",
           label=f"+{MAX_TEST_COST_PCT:g}% limit")
ax.legend()
lo, hi = df["fert_test_prompts"].min(), df["fert_test_prompts"].max()
ax.set_xlim(lo - 0.05 * (hi - lo + 1e-9) - 0.01, hi + 0.05 * (hi - lo + 1e-9) + 0.01)
ax.set_xlabel("fertility on test prompts (tokens / word, lower = better)")
ax.set_title("Test-prompt fertility by mixing strategy")
plt.tight_layout()
plt.savefig(BARS_PATH, dpi=200)
plt.show()
print("Saved:", HEATMAP_PATH)
print("Saved:", BARS_PATH)

## 12. Choose the Final Mix and Retrain It on `FINAL_TRAIN_CHARS`

**Rule:** the test set may be more varied than our estimate, so a tokenizer that only knows the
Supreme Court text is a risk. The candidates are the strategies whose test-prompt fertility is within
`MAX_TEST_COST_PCT` of the best one (an insignificant cost), and among them the winner is the one with
the **lowest average fertility over the 4 held-out domains** (the most robust). `FINAL_STRATEGY`
overrides the rule.

The chosen mix is then **retrained on `FINAL_TRAIN_CHARS` characters** (300M, the same budget as our
current tokenizer; more text gives slightly better merges) and saved to `best_tokenizer/`:
`bpe_tokenizer.json` plus `strategy.json` (shares, characters used, all metrics).

**To use it in a training notebook:** add this notebook's output as a Kaggle input (Add Input ->
Notebook Output) and set `PRETRAINED_BPE_PATH` in the training notebook's configuration to the input
path of `best_tokenizer/bpe_tokenizer.json`; the notebook then skips its own BPE training.

In [ ]:
candidates = df[df["test_cost_pct"] <= MAX_TEST_COST_PCT + 1e-9].sort_values("fert_heldout_macro")
print(f"Candidates (at most {MAX_TEST_COST_PCT:g}% more test-prompt tokens than the best), most robust first:")
print(candidates[["strategy", "fert_test_prompts", "test_cost_pct", "fert_heldout_macro",
                  "fert_heldout_worst"]].round(4).to_string(index=False))
if FINAL_STRATEGY:
    assert FINAL_STRATEGY in strategies, f"FINAL_STRATEGY {FINAL_STRATEGY!r} is not one of {list(strategies)}"
    chosen = FINAL_STRATEGY
    print(f"\nFINAL_STRATEGY set by hand: {chosen}")
else:
    chosen = candidates.iloc[0]["strategy"]
    print(f"\nChosen automatically: {chosen}")
print("  shares: " + ", ".join(f"{d} {100 * s:.1f}%" for d, s in strategies[chosen].items()))

print(f"\nRetraining {chosen} on {FINAL_TRAIN_CHARS / 1e6:,.0f}M characters ...")
final = train_and_evaluate(chosen, FINAL_TRAIN_CHARS, BEST_TOKENIZER_DIR)
print(f"Done in {final['train_minutes']:.1f} min ({final['train_chars'] / 1e6:,.1f}M characters used: "
      + ", ".join(f"{d} {c / 1e6:.1f}M" for d, c in final["chars_used"].items()) + ")")

# compare: final tokenizer vs the same mix at the study budget vs the current tokenizer (SC only)
def summary(r):
    held = [r["metrics"][f"heldout_{d}"]["fertility"] for d in DOMAINS]
    return {"test_prompts": r["metrics"]["test_prompts"]["fertility"], "heldout_macro": sum(held) / len(held),
            "heldout_worst": max(held), **{f"heldout_{d}": r["metrics"][f"heldout_{d}"]["fertility"] for d in DOMAINS},
            "pct_truncated": r["pct_test_prompts_truncated"], "tokens_for_128_words": 128 * r["metrics"]["test_prompts"]["fertility"]}

by_name = {r["strategy"]: r for r in results}
compare = {f"FINAL: {chosen} @ {FINAL_TRAIN_CHARS / 1e6:g}M": summary(final),
           f"{chosen} @ {TOKENIZER_TRAIN_CHARS / 1e6:g}M": summary(by_name[chosen])}
base_name = f"only_{ANCHOR_DOMAIN}"
if base_name in by_name:
    compare[f"{base_name} @ {TOKENIZER_TRAIN_CHARS / 1e6:g}M (current)"] = summary(by_name[base_name])
compare_df = pd.DataFrame(compare).T
print("\nFinal tokenizer vs the study versions (fertility, lower = better):")
print(compare_df.round(4).to_string())
if base_name in by_name:
    b, f_ = summary(by_name[base_name]), summary(final)
    print(f"\nvs {base_name}: test prompts {100 * (f_['test_prompts'] / b['test_prompts'] - 1):+.2f}% tokens | "
          f"average over domains {100 * (f_['heldout_macro'] / b['heldout_macro'] - 1):+.2f}% | "
          f"worst domain {100 * (f_['heldout_worst'] / b['heldout_worst'] - 1):+.2f}%")

with open(os.path.join(BEST_TOKENIZER_DIR, "strategy.json"), "w") as f:
    json.dump({"strategy": chosen, "chosen_by": "FINAL_STRATEGY" if FINAL_STRATEGY else
               f"lowest held-out macro fertility within {MAX_TEST_COST_PCT}% of the best test fertility",
               "shares": strategies[chosen], "chars_used": final["chars_used"], "train_chars": final["train_chars"],
               "vocab_size": final["vocab_size"], "summary": summary(final), "metrics": final["metrics"]}, f, indent=2)
print("\nSaved final tokenizer to:", BEST_TOKENIZER_DIR)

## 13. Final Checks

Fails loudly if anything is wrong: one result and one saved tokenizer per strategy, valid fertility
values (always >= 1, since every word needs at least one token), every output file present, and the
final `best_tokenizer/bpe_tokenizer.json` reloads and gives exactly the fertility measured above.
Then it lists the output folder.

In [ ]:
assert len(results) == len(strategies), f"{len(results)} results for {len(strategies)} strategies"
for r in results:
    assert os.path.exists(os.path.join(TOKENIZERS_DIR, r["strategy"], "bpe_tokenizer.json")), r["strategy"]
    for set_name, m in r["metrics"].items():
        assert math.isfinite(m["fertility"]) and m["fertility"] >= 1.0, (r["strategy"], set_name, m)
for path in (RESULTS_CSV, RESULTS_JSON, HEATMAP_PATH, BARS_PATH, os.path.join(BEST_TOKENIZER_DIR, "bpe_tokenizer.json"),
             os.path.join(BEST_TOKENIZER_DIR, "strategy.json")):
    assert os.path.exists(path), f"missing {path}"

# the saved tokenizer reloads and tokenizes exactly like the trained one
with open(os.path.join(BEST_TOKENIZER_DIR, "bpe_tokenizer.json"), encoding="utf-8") as f:
    data = json.load(f)
reloaded = ByteLevelBPE()
reloaded.itos, reloaded.merges, reloaded.vocab_size, reloaded.symbol_to_id = data["itos"], data["merges"], data["vocab_size"], {}
reloaded._build_ranks()
fert = evaluate_counter(reloaded, eval_sets["test_prompts"])["fertility"]
assert abs(fert - final["metrics"]["test_prompts"]["fertility"]) < 1e-9, (fert, final["metrics"]["test_prompts"])
assert reloaded.vocab_size == VOCAB_SIZE, reloaded.vocab_size

print("Output folder:", OUTPUT_DIR)
for root, dirs, files in os.walk(OUTPUT_DIR):
    for fn in files:
        path = os.path.join(root, fn)
        print(f"  {os.path.relpath(path, OUTPUT_DIR):60s} {os.path.getsize(path) / 2**20:8.2f} MB")
print("\n" + ("SMOKE TEST PASSED: set SMOKE_TEST = False and Save & Run All again." if SMOKE_TEST
              else "ALL CHECKS PASSED: use best_tokenizer/bpe_tokenizer.json in the training notebooks (PRETRAINED_BPE_PATH)."))